In [12]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from astropy import units as u

In [13]:
def toa_calulator(df):
    '''
    Return the toa of a burst
    in the case of multiple components of a single burst, take the middle of the burst
    '''
    if df.shape[0] == 1:
        mjd = df.iloc[0]["toa_bary_tdb_inf_freq"]

    elif df.shape[0] > 1:
        mjd_list = df["toa_bary_tdb_inf_freq"].values
        mjd = (mjd_list[0] + mjd_list[-1]) / 2
 
    return mjd

In [14]:
def toa_utc_calulator(df):
    '''
    Return the toa of a burst
    in the case of multiple components of a single burst, take the middle of the burst
    '''
    if df.shape[0] == 1:
        mjd = df.iloc[0]["toa_mjd_utc"]

    elif df.shape[0] > 1:
        mjd_list = df["toa_mjd_utc"].values
        mjd = (mjd_list[0] + mjd_list[-1]) / 2
 
    return mjd

In [15]:
def peak_sn_calc(df):
    '''
    input: pandas dataframe where each line has the burst properties per component
    Return the maximum value of the snr of all comp
    '''
    
    num_comp = df.shape[0]
    if num_comp == 0:
        print(f"WRONG WRONG : {df}")
    if num_comp == 1:
        peak_sn = df.iloc[0]["peak_snr"]
    elif num_comp > 1:
        peak_list = df["peak_snr"].values
        peak_sn = np.max(peak_list)
    
    return peak_sn,num_comp

In [16]:
def bandwidth_calc(df, telescope):
    '''
    input: pandas dataframe where each line has the burst properties per component
    return the bandwith of a burst
    '''

    obs_band = df.f0_MHz.values[0]
    
    #Pband
    if obs_band == 328.0:
        bw_obs = 54
        
    #Lband
    elif obs_band:
        bw_obs = 128

    if telescope == 'st':
        bw_obs = 98
        
    #We take the absolute value, since the fwidth values can be negative
    #The bandwidth comes from autocorrelation and we define everything larger than FWHM as the BW
    signal_bw = int(abs(df.fwidth_sigma_MHz.values.max()) * 2.355)
    if signal_bw > bw_obs * 0.75:
        signal_bw = bw_obs

    
    return signal_bw, obs_band     

In [17]:
def isotropic_energies(fluences, distance=616):
    "Convert list of fluences to spectral densities"
    "Method based on fluence.py by K.Nimmo"

    #convert from jyms to jys
    fluence_jys = np.array(fluences) * 1e-3

    dis = 1.0 * u.megaparsec
    megap_cm = dis.to(u.cm).value

    #mpc to cm
    distance_lum_cm = megap_cm*distance

    #redshift correction, see evernote for details
    #redshift is from: Snelders et al
    #distance is also from Ravi et al, FAST converts themselves based on PLANCK 2016
    #link: https://ui.adsabs.harvard.edu/abs/2025ApJ...992L..35B/abstract
    z = 0.130287
    red_cor = 1 / (1 + z)**2

    energy_iso = fluence_jys * 4*np.pi*(distance_lum_cm**2) * 1e-23 * red_cor
    #print(energy_iso)

    return energy_iso

In [18]:
def speclum(fluence, ontime, distance=616, z=0.130287):
    '''
    fluence assumed in Jy~ms
    ontime assumed in ms
    distance assumed in Mpc
    '''
    # actual ontime or total width? Width gives a lover limit on the luminosity.
    
    #convert Jy ms to J s; and ms to s
    fluence_jys = fluence*1e-3
    ontime /= 1e3
    
    #convert Mpc to cm
    distance_lum_cm = 3.086e24*distance
    energy_iso= fluence_jys*4*np.pi*(distance_lum_cm**2)*1e-23 / (1+z)**(2)
    lum_spec = energy_iso/ontime
    
    return lum_spec

In [19]:
def fluence_looper(df_fluence):
    """
    function to sum over the fluences components of bursts
    
    Input: The burst_csv file with all information, spc/sfxc/scale
    This is needed because we return a df with fluence (spc) and toa (sfxc)
    
    return: pandas dataframe with: id/fluence/telescope/c_freq/toa/width_ms
    """

    #the burst names to loop over
    exps = df_fluence['id'].unique()

    #Create a list with the telescope names for easy filtering later on
    exps_tel = [i.split('-')[1] for i in exps]

    #loop over every burst to sum the fluence
    info_list = []
    for exp, telescope in zip(exps, exps_tel):
        print(exp)
        #Fluence: Making a small temp df for each exp, only taking the spc values
        if telescope != 'st':
            fluence_df_exp_temp = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'spc')]
            
            #TOA
            fluence_df_exp_temp_mjd = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'sfxc')]
            toa = toa_calulator(fluence_df_exp_temp_mjd) 
            toa_utc = toa_utc_calulator(fluence_df_exp_temp_mjd) 
                
        elif telescope == 'st':
            fluence_df_exp_temp = df_fluence[df_fluence['id'] == exp]
            
            #TOA
            toa = toa_calulator(fluence_df_exp_temp) 
            toa_utc = toa_utc_calulator(fluence_df_exp_temp)
            
        #Calculate the total fluence
        fluence = sum(fluence_df_exp_temp['fluence_jyms'].values)
        
        #Calculate isotropic energies
        energies_isotropic = isotropic_energies(fluence)
        
        #Calculate the max SN of a burst
        peak_sn, num_comp = peak_sn_calc(fluence_df_exp_temp)
    
        #Calculate the bandwith
        signal_bw, cent_freq = bandwidth_calc(fluence_df_exp_temp, telescope)
        
        #width in ms
        width_t = fluence_df_exp_temp.end_acf_range.values.max() - fluence_df_exp_temp.begin_acf_range.values.min()

        #Calculate the spectral luminosity
        spel_lum = speclum(fluence, width_t, distance=362.4, z=0.0771)
        
        #Since the F0 is the same for all components, take all components, take the first entry
        cent_freq = fluence_df_exp_temp['f0_MHz'].values[0]
        
        #Add to the list
        info_list.append([toa, peak_sn, fluence, num_comp, width_t, energies_isotropic, spel_lum, \
                          signal_bw, cent_freq, toa_utc])                          
        
    #Convert the fluence list to array
    info_list_arr = np.asarray(info_list)
    
    #Convert the info to dataframe
    new_data = {'id': exps, 'station': exps_tel, 'toa': info_list_arr[:,0], \
                'peak_sn': info_list_arr[:,1], 'fluence': info_list_arr[:,2], \
                'Number of components': info_list_arr[:,3],\
                'width_ms': info_list_arr[:,4], \
                'spectral density': info_list_arr[:,5],\
                'spectral luminosity': info_list_arr[:,6], \
                'bandwidth': info_list_arr[:,7], \
                'central frequency': info_list_arr[:,8],\
                'toa_utc': info_list_arr[:,9]}
    
    df_new_fluence = pd.DataFrame(data=new_data)

    #add only the index number as a seperate column for potential later use
    df_new_fluence['burst-index'] = df_new_fluence['id'].str.split('-').str[0].str[1:].astype(int)
    
    return df_new_fluence

In [20]:
def df_to_small_table(file):
    """
    Function to load in the burst.csv file and return the shortend version of the .csv file
    """

    #load in the main csv file
    fluence_df = pd.read_csv(file, index_col=0, header=0, na_values='NA', engine='python').reset_index(drop=True)
    fluence_df.sort_values(by=['id'])
    fluence_df = fluence_df[~fluence_df['id'].isin(['B88-tr', 'B157-tr'])]
    print(set(fluence_df["experiment"].values))
    
    df_combined = fluence_looper(fluence_df)    
    df_combined.sort_values(by=['id'])

    #Resort and reset the df
    df_combined = df_combined.sort_values(by=['id'])
    df_combined = df_combined.reset_index(drop=True)
    
    #Setting the formatting of the numbers correct
    #df_combined["Number of components"] = df_combined["Number of components"].astype(int)
    df_combined["bandwidth"] = df_combined["bandwidth"].astype(int)
    df_combined["central frequency"] = df_combined["central frequency"].astype(int)
    
    df_combined['fluence_err'] = pd.to_numeric(df_combined['fluence'] * 0.2, errors='coerce').round(2)
    
    #Minimum spectral energy is on the order of 1e30 // calculated with the .min()
    df_combined['spectral density'] /= 1e30
    df_combined['spec_den_err'] = df_combined['spectral density'] * 0.2
    df_combined['pectral density'] = df_combined['spectral density'].round(2)
    df_combined['spec_den_err'] = df_combined['spec_den_err'].round(2)
    
    #Minimum spectral lum is on the order of 1e31 // calculated with the .min()
    df_combined['spectral luminosity'] /= 1e31
    df_combined['spec_lum_err'] = df_combined['spectral luminosity'] * 0.2
    df_combined['spectral luminosity'] = df_combined['spectral luminosity'].round(2)
    df_combined['spec_lum_err'] = df_combined['spec_lum_err'].round(2)

    df_combined['peak_sn'] = pd.to_numeric(df_combined['peak_sn'], errors='coerce').round(2)
    df_combined['width_ms'] = pd.to_numeric(df_combined['width_ms'], errors='coerce').round(2)
    df_combined['fluence'] = pd.to_numeric(df_combined['fluence'], errors='coerce').round(2)

    df_combined['peak_sn'] = df_combined['peak_sn'].round(2)
    df_combined['width_ms'] = df_combined['width_ms'].round(2)
    df_combined['fluence'] = df_combined['fluence'].round(2)

    df_paper = df_combined[['id', 'station', 'toa', 'toa_utc', 'peak_sn', 'fluence', 'fluence_err', "Number of components",\
                            'width_ms', 'spectral density', 'spec_den_err', 'spectral luminosity', 'spec_lum_err',\
                            'bandwidth', 'central frequency']]

    #Adding the Dwingeloo burst by hand
    # new_row = pd.DataFrame([{
    # 'id': 'B06-dw',
    # 'station': 'dw',
    # 'toa': 60537.8884462584246648, 
    # 'peak_sn': np.NaN,
    # 'fluence': np.NaN,
    # 'fluence_err': np.NaN,
    # 'width_ms': 1,
    # 'bandwidth' : 200
    # }])
    #df_paper = pd.concat([df_paper, new_row], ignore_index=True)

    df_paper = df_paper.sort_values(by=['id']).reset_index(drop=True)
    name_csv = "R147_table_v1.csv"
    #df_paper.to_csv(name_csv, header=True, index=True, na_rep='NA')
    display(df_paper)

    return 

In [21]:
df_to_small_table("stockert_bursts_r147.csv")
#display(fluence_df)

{'stocke'}
B03-st
B05-st
B06-st
B08-st
B15-st
B16-st
B20-st
B21-st
B22-st
B23-st
B24-st
B30-st
B31-st
B35-st
B36-st
B38-st
B39-st
B40-st
B41-st
B42-st
B44-st
B04-st
B34-st
B45-st
B56-st
B49-st
B50-st
B51-st
B53-st
B54-st
B55-st
B57-st
B58-st
B59-st
B60-st
B61-st
B62-st
B66-st
B73-st
B74-st
B75-st
B76-st
B80-st
B87-st
B88-st
B100-st
B101-st
B102-st
B103-st
B104-st
B107-st
B109-st
B113-st
B114-st
B115-st
B116-st
B117-st
B118-st
B119-st
B121-st
B123-st
B125-st
B131-st
B167-st
B37-st
B43-st
B120-st
B124-st
B127-st


,id,station,toa,toa_utc,peak_sn,fluence,fluence_err,Number of components,width_ms,spectral density,spec_den_err,spectral luminosity,spec_lum_err,bandwidth,central frequency
0,B03-st,st,60357.339134,60357.343782,8.60,20.15,4.03,1.0,3.71,7.160011,1.43,73.50,14.70,98,1381
1,B04-st,st,60357.433876,60357.438523,6.00,25.10,5.02,1.0,9.61,8.920358,1.78,35.38,7.08,53,1381
2,B05-st,st,60361.542887,60361.547509,5.21,35.49,7.10,1.0,7.86,12.610859,2.52,61.13,12.23,98,1381
3,B06-st,st,60366.341592,60366.346149,5.26,16.72,3.34,1.0,3.50,5.941981,1.19,64.81,12.96,98,1381
4,B08-st,st,60369.331499,60369.335997,4.72,12.74,2.55,1.0,3.06,4.526881,0.91,56.43,11.29,98,1381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,B75-st,st,60381.202622,60381.206745,9.19,27.39,5.48,1.0,4.59,9.733133,1.95,80.88,16.18,63,1381
65,B76-st,st,60381.215036,60381.219158,5.96,30.41,6.08,1.0,11.36,10.806404,2.16,36.27,7.25,98,1381
66,B80-st,st,60381.300292,60381.304411,10.71,132.58,26.52,1.0,31.68,47.115952,9.42,56.70,11.34,98,1381
67,B87-st,st,60381.445612,60381.449725,4.52,30.10,6.02,1.0,6.99,10.697869,2.14,58.34,11.67,73,1381


In [22]:
df_to_small_table("hyperflash_westerbork_L.csv")
#display(fluence_df)

{'pcn254', 'p55084', 'p55092', 'p47031', 'p55097', 'p55186', 'p47018', 'p55179', 'p55293', 'p47066', 'p55183', 'p55195', 'p47085', 'p47021', 'p55173', 'p47021-2', 'p47019', 'p55179-2', 'p47065', 'p55197', 'p47057', 'p55299', 'p47020', 'p55158', 'p55182'}
B02-wb
B09-wb
B11-wb
B12-wb
B13-wb
B14-wb
B18-wb
B19-wb
B30-wb
B33-wb
B32-wb
B25-wb
B26-wb
B28-wb
B29-wb
B110-wb
B112-wb
B135-wb
B136-wb
B137-wb
B139-wb
B140-wb
B142-wb
B143-wb
B144-wb
B145-wb
B146-wb
B147-wb
B151-wb
B158-wb
B160-wb
B163-wb
B173-wb
B177-wb
B130-wb
B123-wb
B126-wb
B127-wb
B156-wb


,id,station,toa,toa_utc,peak_sn,fluence,fluence_err,Number of components,width_ms,spectral density,spec_den_err,spectral luminosity,spec_lum_err,bandwidth,central frequency
0,B02-wb,wb,60355.639527,60355.644167,5.26,16.66,3.33,1.0,14.34,5.922223,1.18,15.75,3.15,128,1271
1,B09-wb,wb,60369.366278,60369.370769,4.43,7.34,1.47,1.0,22.53,2.608990,0.52,4.42,0.88,77,1271
2,B11-wb,wb,60370.289589,60370.294059,4.85,8.67,1.73,1.0,11.26,3.080679,0.62,10.43,2.09,128,1271
3,B110-wb,wb,60387.363333,60387.367176,5.88,10.14,2.03,1.0,2.56,3.605125,0.72,53.68,10.74,128,1271
4,B112-wb,wb,60387.529496,60387.533331,5.73,23.71,4.74,1.0,4.86,8.425037,1.69,66.03,13.21,128,1271
5,B12-wb,wb,60370.416514,60370.420980,6.97,15.48,3.10,1.0,7.17,5.499681,1.10,29.25,5.85,128,1271
6,B123-wb,wb,60424.432518,60424.433747,5.44,6.09,1.22,1.0,3.07,2.164449,0.43,26.86,5.37,78,1271
7,B126-wb,wb,60432.385082,60432.385605,8.14,36.38,7.28,1.0,10.75,12.928492,2.59,45.84,9.17,128,1271
8,B127-wb,wb,60433.203153,60433.203602,4.14,21.31,4.26,1.0,10.75,7.573418,1.51,26.85,5.37,94,1271
9,B13-wb,wb,60370.447553,60370.452019,12.83,42.40,8.48,1.0,5.63,15.067215,3.01,101.99,20.40,93,1271


In [23]:
df_to_small_table("hyperflash_torun_L.csv")
#display(fluence_df)

{'rn1l82', 'rn1l87', 'rn1l81', 'rn1l11', 'rn1l13', 'rn1l90', 'rn1l33', 'rn1l11-2', 'rn1l83', 'rn1l59', 'rn1l12', 'rn1l58', 'rn1l10', 'rn1l67', 'rn1l64', 'rn1l62', 'rn1l84', 'rn1l65', 'rn1l40', 'rn1l70', 'rn1l89'}
B60-tr
B61-tr
B64-tr
B65-tr
B66-tr
B68-tr
B58-tr
B85-tr
B87-tr
B90-tr
B134-tr
B137-tr
B138-tr
B139-tr
B141-tr
B149-tr
B148-tr
B154-tr
B155-tr
B152-tr
B153-tr
B158-tr
B163-tr
B164-tr
B165-tr
B166-tr
B167-tr
B168-tr
B169-tr
B174-tr
B176-tr
B180-tr
B110-tr
B122-tr


,id,station,toa,toa_utc,peak_sn,fluence,fluence_err,Number of components,width_ms,spectral density,spec_den_err,spectral luminosity,spec_lum_err,bandwidth,central frequency
0,B110-tr,tr,60387.363329,60387.367173,12.65,9.63,1.93,1.0,4.10,3.423108,0.68,31.86,6.37,128,1418
1,B122-tr,tr,60398.434455,60398.437673,9.50,13.44,2.69,1.0,7.30,4.777341,0.96,24.96,4.99,128,1418
2,B134-tr,tr,60563.957078,60563.951197,13.30,20.84,4.17,1.0,6.66,7.404960,1.48,42.41,8.48,72,1444
3,B137-tr,tr,60586.957653,60586.952995,33.47,113.18,22.64,1.0,14.08,40.221904,8.04,108.90,21.78,70,1444
4,B138-tr,tr,60654.792675,60654.794005,4.48,6.79,1.36,1.0,10.24,2.414543,0.48,8.99,1.80,128,1444
5,B139-tr,tr,60658.718967,60658.720637,14.53,21.30,4.26,1.0,2.82,7.570416,1.51,102.48,20.50,128,1444
6,B141-tr,tr,60674.669724,60674.672635,363.47,805.41,161.08,1.0,9.34,286.228245,57.25,1167.75,233.55,128,1444
7,B148-tr,tr,60681.571168,60681.574531,3.42,28.14,5.63,1.0,3.58,9.998717,2.00,106.35,21.27,0,1406
8,B149-tr,tr,60681.665805,60681.669174,7.91,42.90,8.58,1.0,32.77,15.246928,3.05,17.74,3.55,128,1444
9,B152-tr,tr,60683.542983,60683.546465,12.23,33.93,6.79,1.0,11.78,12.058190,2.41,39.04,7.81,78,1444


In [24]:
df_to_small_table("hyperflash_onsala_L.csv")

{'p47030', 'p49035', 'p47027', 'p47024-2', 'p47026', 'p47038', 'p47025-2', 'p47025', 'p47064', 'p47028', 'p47068', 'p47024'}
B63-o8
B65-o8
B67-o8
B68-o8
B70-o8
B71-o8
B72-o8
B56-o8
B57-o8
B79-o8
B80-o8
B82-o8
B83-o8
B84-o8
B74-o8
B85-o8
B86-o8
B87-o8
B88-o8
B89-o8
B90-o8
B91-o8
B92-o8
B75-o8
B76-o8
B77-o8
B78-o8
B96-o8
B97-o8
B98-o8
B99-o8
B100-o8
B101-o8
B93-o8
B94-o8
B95-o8
B102-o8
B103-o8
B104-o8
B106-o8
B107-o8
B108-o8
B120-o8
B128-o8
B162-o8
B73-o8
B125-o8
WRONG WRONG : Empty DataFrame
Columns: [experiment, dish, scan, component, src, t0_ms, f0_MHz, toa_loc, terr_ms, ferr_MHz, gauss2d_twidth_ms, gauss2d_fwidth_MHz, begin_acf_range, end_acf_range, twidth_sigma_ms, fwidth_sigma_MHz, twidtherr_ms, fwidtherr_MHz, off_range_t0, off_range_t1, fluence_jyms, peak_snr, peak_flux_jy, fluence_tot_jyms, scint_bw_MHz, scint_bw_err_MHz, energy_iso_erg_per_hz, spectral_lum_erg_per_s_per_hz, id, sefd_jy, distance_Mpc, dm, toa_mjd_utc, ref_freq_MHz, toa_bary_tdb_inf_freq]
Index: []

[0 rows x 35 c

UnboundLocalError: cannot access local variable 'peak_sn' where it is not associated with a value

In [13]:
# def max_values():
    
#     # full_db = pd.read_csv('burst_stats_r117.csv', sep=",", header=0, index_col=0, na_values='NA', engine='python')
#     # idx_max_jy = full_db['peak_flux_jy'].idxmax()
#     # Filter_df = full_db[full_db.index.isin([idx_max_jy])]
#     # display(Filter_df)

#     table_db = pd.read_csv('FRB20240619D_HyperFlash_table.csv', sep=",", header=0, index_col=0, na_values='NA', engine='python')

#     # idx_max_jy = table_db['fluence'].idxmax()
#     # Filter_df = table_db[table_db.index.isin([idx_max_jy])]
#     # display(Filter_df)

#     # db_l = table_db[table_db['bandwidth'] != 54]
#     # idx_max_jy = db_l['fluence'].idxmax()
#     # Filter_df = db_l[db_l.index.isin([idx_max_jy])]
#     # display(Filter_df)

#     db_l = table_db[table_db['bandwidth'] != 54]
#     widths = db_l["width_ms"].values
#     print(np.median(widths))
#     print(np.min(widths))
#     print(np.max(widths))

#     # db_l = table_db[table_db['bandwidth'] == 54]
#     # widths = db_l["width_ms"].values
#     # print(np.median(widths))
#     # print(np.min(widths))
    
#     return
# #max_values()

In [14]:
# def calc_time_delay(freq, dm, dmconst):
#     """
#     Input:
#         freq (MHz), float
#         dm (pc/cc), float
#         dmconst (GHz^2 cm^3 pc^-1 ms), float
#     Returns:
#         The time difference between infinite frequency and freq
#         in SECONDS"""
#     return dmconst * 10**6. * dm * freq**-2. / 1000.

In [12]:
# from astropy.coordinates import EarthLocation
# from astropy import coordinates as coord
# import astropy.units as u
# from astropy.time import Time

# def loc2bary(mjd, dm_const=1/0.241):

#     #Via Dante
#     ref_freq = 1738

#     #DM used for all bursts in NRT analysis
#     dm = 464.86

#     #Coordinates of NRT
#     source = coord.SkyCoord('19:49:29.21', '-25:12:49.40', unit=(u.hourangle, u.deg))

#     #NRT via D. Hewitt
#     station = EarthLocation.from_geocentric(x=4324165.81 * u.m,
#                                             y=165927.11 * u.m,
#                                             z=4670132.83 * u.m)

#     #Define the local MJD and the location
#     localTOA = Time(mjd, format='mjd', scale='utc', location=station)    
#     delay_to_inf = calc_time_delay(ref_freq, dm, dm_const) * u.s  # in seconds  
#     delay_to_bary = localTOA.light_travel_time(source, 'barycentric')  # in TDB seconds, to be added
#     baryTOA = localTOA - delay_to_inf + delay_to_bary
    
#     return baryTOA.tdb.value

# #loc2bary(mjd=[60511.987776,60511.987776], dm_const=1/0.241)

In [10]:
# def nancay_rmkt(csv_file, mjd_limit=True):
#     """
#     Imports, processes, and merges fluence and rotation measure (RM) data 
#     for the Nançay Radio Telescope (NRT) burst sample.

#     This function:
#     - Loads a CSV file with burst properties.
#     - Renames and standardizes relevant columns.
#     - Converts local MJDs to barycentric TDB using `loc2bary`.
#     - Computes fluence uncertainties as 20% of the measured fluence.
#     - Rounds and cleans various fields.
#     - Loads a separate CSV with RM measurements and merges it with the main DataFrame 
#       based on rounded MJD values.
#     - Reorders and filters columns.
#     - Saves the resulting database to a new CSV file.

#     Parameters
#     ----------
#     csv_file : str
#         Path to the CSV file containing the NRT burst data.
#     mjd_limit : bool, optional
#         Currently unused; placeholder for future functionality to filter on MJD.

#     Returns
#     -------
#     None
#         The function displays the final DataFrame and saves it to a CSV file.
#     """
    
#     nancay_df = pd.read_csv(csv_file, on_bad_lines='skip', header=0, sep=';')
#     #print(nancay_df.columns)
    
#     nancay_df.rename(columns={'MJD_at_peak': 'MJD_local'}, inplace=True)
#     nancay_df.rename(columns={'burst_name': 'burst_name_internal'}, inplace=True)
#     nancay_df.rename(columns={'event_duration_ms': 'width_ms'}, inplace=True)
    
#     mjd_local_vals = nancay_df["MJD_local"].values
#     mjd_bary_vals = loc2bary(mjd_local_vals)

#     nancay_df["MJD_bary_tdb"] = mjd_bary_vals

#     # Round the 'fluence_Jyms' column to two decimal places
#     nancay_df['fluence_Jyms'] = nancay_df['fluence_Jyms'].round(3)
#     nancay_df['width_ms'] = nancay_df['fluence_Jyms'].round(2)
#     nancay_df['spectral_extent_MHz'] = nancay_df['spectral_extent_MHz'].astype(int)
    
#     # Create a new column 'fluence_Jyms_error' as 20% of 'fluence_Jyms'
#     nancay_df['fluence_Jyms_error'] = nancay_df['fluence_Jyms'] * 0.20
    
#     # Select only the desired columns
#     columns_to_keep = ['MJD_bary_tdb', 'peak_flux', 'fluence_Jyms', 'fluence_Jyms_error', 'width_ms', 'spectral_extent_MHz','burst_name_internal', 'MJD_local']
#     df_filtered = nancay_df[columns_to_keep].sort_values(by='MJD_bary_tdb', ascending=True).reset_index(drop=True)
    
#     ###
#     #Adding the RM information in to the db 
#     ###

#     ##Loading in the csv file
#     rm_df = pd.read_csv("NRT_RM_values_and_errors.csv", header=0, index_col=0)

#     #Now we introduce the mjd_rounded variable, so we can easily merge the dfs with each other
#     df_filtered['mjd_rounded'] = df_filtered['MJD_local'].astype(float).round(6)
#     rm_df['mjd_rounded'] = rm_df['MJD'].astype(float).round(6)

#     #Using left, we keep all lines and Nans are introduced
#     merged = pd.merge(df_filtered, rm_df, on='mjd_rounded', how='left')

#     ##Now we drop some columns
#     df_nrt = merged.drop(columns=['MJD','mjd_rounded', 'RM_pcm60499', 'RM_pcm60598'])

#     # Create custom index column with prefix 'B' and numbers from B001 to B206
#     df_nrt['index'] = ['B' + str(i).zfill(3) for i in range(1, 207)]
    
#     # Move the index column to the front (optional)
#     df_nrt = df_nrt[['index'] + [col for col in df_nrt.columns if col != 'index']]

#     desired_order = [
#                 'index','MJD_bary_tdb', 'peak_flux', 'fluence_Jyms', 'fluence_Jyms_error',
#                 'width_ms', 'spectral_extent_MHz',
#                 'fit_RM_pcm60499', 'fit_RM_err_pcm60499',
#                 'fit_RM_pcm60598', 'fit_RM_err_pcm60598',
#                 'MJD_local', 'burst_name_internal'
#                     ]
#     df_nrt = df_nrt[desired_order]

#     display(df_nrt)
#     #Saving the db
#     name_csv = "NRT_burst_db_bary_pol_v2.csv"
#     #df_nrt.to_csv(name_csv, header=True, index=True, na_rep='NA')
    
#     return df_nrt

# df_nrt = nancay_rmkt(csv_file="/home/omar/git/frb20240619d-single-dish-analysis/dbs/nrt/nrt_burst_db_v2.csv",\
#                                  mjd_limit=False)

In [11]:
# df_nrt.sort_values(by=["fluence_Jyms"])